# Construction du dataset historique

Les quatre fichiers sources sont assemblés (une ligne par pays, année et culture, sur 1990-2013),
puis nettoyés.

| Étape | Résultat |
|---|---|
| 1. État après jointures | 22 679 lignes, 168 pays, avec des valeurs manquantes (non sauvegardé) |
| 2. Nettoyage | 16 357 lignes, 117 pays, aucune valeur manquante : `data/processed/crop_yield_clean.csv` |

| Choix | Raison |
|---|---|
| Période 1990-2013 | années communes aux quatre fichiers |
| Jointures sur le code ISO3 | une jointure sur le nom du pays perd 18 pays, dont les États-Unis, la Chine et la Russie |
| `China, mainland` gardé | les autres fichiers traitent la Chine continentale |
| `temp.csv` : doublons retirés, puis moyenne par pays et par année | sinon les lignes de rendement seraient dupliquées à la jointure |
| Pluie manquante : valeur du pays reprise | la pluie d'un pays est la même chaque année |
| Pays sans aucune température ou sans aucun pesticide retirés | aucune valeur du pays à reprendre, rien n'est inventé |

# Imports

In [1]:
import pandas as pd

from agritech import geo
from agritech.config import PATHS

# Sources

In [2]:
DEBUT, FIN = 1990, 2013
files_dir = PATHS.data_crop_yield_prediction

df_yield = pd.read_csv(files_dir / "yield.csv")
df_temp = pd.read_csv(files_dir / "temp.csv")
df_rainfall = pd.read_csv(files_dir / "rainfall.csv")
df_pesticides = pd.read_csv(files_dir / "pesticides.csv")

# rainfall.csv a une espace initiale dans l'en-tête de sa colonne pays
df_rainfall.columns = df_rainfall.columns.str.strip()

for nom, df in [("yield", df_yield), ("temp", df_temp),
                ("rainfall", df_rainfall), ("pesticides", df_pesticides)]:
    print(f"{nom:12} {df.shape[0]:6} lignes x {df.shape[1]} colonnes")

yield         56717 lignes x 12 colonnes
temp          71311 lignes x 3 colonnes
rainfall       6727 lignes x 3 colonnes
pesticides     4349 lignes x 7 colonnes


# Préparation des sources

In [3]:
def ajoute_iso3(df, colonne_pays):
    """Ajoute une colonne iso3 à partir du nom de pays."""
    correspondances = geo.vers_iso3(df[colonne_pays].astype(str).unique())
    return df.assign(iso3=df[colonne_pays].map(correspondances))

## Rendement

`China` regroupe la Chine continentale, Taïwan, Hong Kong et Macao. Les autres fichiers traitent ces
territoires à part : on garde donc `China, mainland`.

In [4]:
rendement = (
    df_yield[df_yield["Area"] != "China"]
    .query("@DEBUT <= Year <= @FIN")
    .pipe(ajoute_iso3, "Area")
    .rename(columns={"Area": "area", "Year": "year", "Item": "crop"})
    .assign(yield_t_ha=lambda d: d["Value"] / 10_000)   # hg/ha -> t/ha
    [["iso3", "area", "year", "crop", "yield_t_ha"]]
)

rendement.head()

,iso3,area,year,crop,yield_t_ha
29,AFG,Afghanistan,1990,Maize,1.7582
30,AFG,Afghanistan,1991,Maize,1.6800
31,AFG,Afghanistan,1992,Maize,1.5000
32,AFG,Afghanistan,1993,Maize,1.6786
33,AFG,Afghanistan,1994,Maize,1.6667


## Température

Doublons exacts retirés, puis moyenne simple des relevés d'un pays pour une année.

In [5]:
temperature = (
    df_temp.drop_duplicates()
    .query("@DEBUT <= year <= @FIN")
    .pipe(ajoute_iso3, "country")
    .groupby(["iso3", "year"], as_index=False)["avg_temp"].mean()
)

temperature.head()

,iso3,year,avg_temp
0,AFG,1990,15.45
1,AFG,1991,14.57
2,AFG,1992,14.35
3,AFG,1993,14.96
4,AFG,1994,14.94


## Pluie

La colonne est lue comme du texte à cause de quelques valeurs `..`, qui deviennent des valeurs
manquantes.

In [6]:
pluie = (
    df_rainfall
    .assign(rain_mm=lambda d: pd.to_numeric(d["average_rain_fall_mm_per_year"], errors="coerce"))
    .query("@DEBUT <= Year <= @FIN")
    .pipe(ajoute_iso3, "Area")
    .rename(columns={"Year": "year"})
    [["iso3", "year", "rain_mm"]]
)

pluie.head()

,iso3,year,rain_mm
4,AFG,1990,327.0
5,AFG,1991,327.0
6,AFG,1992,327.0
7,AFG,1993,327.0
8,AFG,1994,327.0


## Pesticides

Une seule ligne par pays et par année : rien à regrouper.

In [7]:
pesticides = (
    df_pesticides
    .query("@DEBUT <= Year <= @FIN")
    .pipe(ajoute_iso3, "Area")
    .rename(columns={"Year": "year", "Value": "pesticides_t"})
    [["iso3", "year", "pesticides_t"]]
)

pesticides.head()

,iso3,year,pesticides_t
0,ALB,1990,121.0
1,ALB,1991,121.0
2,ALB,1992,121.0
3,ALB,1993,121.0
4,ALB,1994,201.0


# Contrôles avant les jointures

In [8]:
sources = {
    "rendement": (rendement, ["iso3", "year", "crop"]),
    "température": (temperature, ["iso3", "year"]),
    "pluie": (pluie, ["iso3", "year"]),
    "pesticides": (pesticides, ["iso3", "year"]),
}

for nom, (df, cle) in sources.items():
    sans_iso3 = df["iso3"].isna().sum()
    valides = df.dropna(subset=["iso3"])
    doublons = valides.duplicated(cle).sum()
    print(f"{nom:12} {len(df):6} lignes | {valides['iso3'].nunique():3} pays "
          f"| sans ISO3 : {sans_iso3:5} | doublons sur {'+'.join(cle)} : {doublons}")
    assert doublons == 0, f"{nom} : la clé {cle} n'est pas unique"

rendement     25848 lignes | 168 pays | sans ISO3 :  3169 | doublons sur iso3+year+crop : 0
température    3192 lignes | 133 pays | sans ISO3 :     0 | doublons sur iso3+year : 0
pluie          4991 lignes | 170 pays | sans ISO3 :  1081 | doublons sur iso3+year : 0
pesticides     3860 lignes | 144 pays | sans ISO3 :   506 | doublons sur iso3+year : 0


**Observations :**

- Chaque source a une seule ligne par clé : les jointures ne dupliqueront pas de lignes.
- Les lignes sans code ISO3 (micro-États, territoires, pays disparus) sont écartées.

# État après jointures

In [9]:
apres_jointures = rendement.dropna(subset=["iso3"])
print(f"base rendement  {len(apres_jointures)} lignes")

for nom, source in [("température", temperature), ("pluie", pluie), ("pesticides", pesticides)]:
    avant = len(apres_jointures)
    apres_jointures = apres_jointures.merge(source.dropna(subset=["iso3"]), on=["iso3", "year"], how="left")
    print(f"+ {nom:12} {avant} -> {len(apres_jointures)} lignes")
    assert len(apres_jointures) == avant, f"lignes dupliquées par la jointure {nom}"

base rendement  22679 lignes
+ température  22679 -> 22679 lignes
+ pluie        22679 -> 22679 lignes
+ pesticides   22679 -> 22679 lignes


## Contrôles

In [10]:
CLE = ["iso3", "year", "crop"]

print(f"lignes     : {len(apres_jointures)}")
print(f"pays       : {apres_jointures['iso3'].nunique()}")
print(f"cultures   : {apres_jointures['crop'].nunique()}")
print(f"années     : {apres_jointures['year'].min()} - {apres_jointures['year'].max()} "
      f"({apres_jointures['year'].nunique()} distinctes)")
print(f"clé unique : {not apres_jointures.duplicated(CLE).any()}")
print(f"colonnes   : {list(apres_jointures.columns)}")
print("\nvaleurs manquantes (%) :")
print((apres_jointures.isna().mean() * 100).round(1).to_string())

assert len(apres_jointures) == 22_679
assert apres_jointures["iso3"].nunique() == 168
assert not apres_jointures.duplicated(CLE).any()

lignes     : 22679
pays       : 168
cultures   : 10
années     : 1990 - 2013 (24 distinctes)
clé unique : True
colonnes   : ['iso3', 'area', 'year', 'crop', 'yield_t_ha', 'avg_temp', 'rain_mm', 'pesticides_t']

valeurs manquantes (%) :
iso3             0.0
area             0.0
year             0.0
crop             0.0
yield_t_ha       0.0
avg_temp        19.5
rain_mm          5.0
pesticides_t    13.1


**Observations :**

- 22 679 lignes après les jointures : 168 pays, 10 cultures, 1990-2013.
- Une seule ligne par pays, année et culture : les jointures n'ont dupliqué aucune ligne.
- Il manque encore des valeurs : température 19,5 %, pesticides 13,1 %, pluie 5,0 %.

# Nettoyage

Deux règles, sans inventer de valeurs :

1. **Pluie** : la pluie d'un pays est la même chaque année. Une année sans valeur reprend donc la
   valeur connue du pays.
2. **Température et pesticides** : un pays sans aucune valeur sur 1990-2013 est retiré.

## Où sont les valeurs manquantes ?

Quelques années manquantes dans un pays, ou pays entièrement absent d'une source ?

In [11]:
def couverture(df, colonne):
    """Compte les pays sans aucune valeur et ceux à qui il manque seulement quelques années."""
    par_pays = df.groupby("iso3")[colonne].agg(["count", "size"])
    aucun = par_pays[par_pays["count"] == 0]
    partiel = par_pays[(par_pays["count"] > 0) & (par_pays["count"] < par_pays["size"])]
    return pd.Series({
        "lignes manquantes": df[colonne].isna().sum(),
        "pays sans aucune valeur": len(aucun),
        "lignes de ces pays": aucun["size"].sum(),
        "pays avec quelques années manquantes": len(partiel),
    })


pd.DataFrame({colonne: couverture(apres_jointures, colonne) for colonne in ["avg_temp", "rain_mm", "pesticides_t"]})

,avg_temp,rain_mm,pesticides_t
lignes manquantes,4415,1130,2975
pays sans aucune valeur,36,1,24
lignes de ces pays,4415,177,2975
pays avec quelques années manquantes,0,163,0


**Observations :**

- Température : 36 pays n'ont aucune valeur, aucun autre pays n'a de trou.
- Pesticides : 24 pays n'ont aucune valeur, aucun autre pays n'a de trou.
- Pluie : 1 pays sans aucune valeur et 163 pays avec quelques années manquantes.

## Pluie : reprise de la valeur du pays

Seules les valeurs manquantes sont complétées, avec la valeur connue du même pays.

In [12]:
# rain_mm a une seule valeur par pays : c'est sa valeur de pluie fixe dans le fichier
print("valeurs distinctes de rain_mm par pays :", apres_jointures.groupby("iso3")["rain_mm"].nunique().max(), "au maximum")
assert apres_jointures.groupby("iso3")["rain_mm"].nunique().max() == 1

lignes_sans_pluie = apres_jointures[apres_jointures["rain_mm"].isna()]
print("\nlignes sans pluie par année :")
print(lignes_sans_pluie["year"].value_counts().sort_index().to_dict())
print("\nlignes sans pluie hors 2003, par pays :")
print(lignes_sans_pluie[lignes_sans_pluie["year"] != 2003].groupby("area")["year"].agg(["size", "min", "max"]).to_string())

valeurs distinctes de rain_mm par pays : 1 au maximum

lignes sans pluie par année :
{1990: 11, 1991: 10, 1992: 8, 1993: 8, 1994: 8, 1995: 8, 1996: 7, 1997: 8, 1998: 8, 1999: 8, 2000: 8, 2001: 7, 2002: 8, 2003: 954, 2004: 7, 2005: 7, 2006: 7, 2007: 7, 2008: 6, 2009: 6, 2010: 6, 2011: 7, 2012: 8, 2013: 8}

lignes sans pluie hors 2003, par pays :
               size   min   max
area                           
Bahamas           6  1990  1991
New Caledonia   170  1990  2013


In [13]:
pluie_du_pays = apres_jointures.groupby("iso3")["rain_mm"].first()   # first() ignore les valeurs manquantes
a_completer = apres_jointures["rain_mm"].isna() & apres_jointures["iso3"].map(pluie_du_pays).notna()

pluie_completee = apres_jointures.copy()
pluie_completee.loc[a_completer, "rain_mm"] = pluie_completee.loc[a_completer, "iso3"].map(pluie_du_pays)

completees_par_annee = pluie_completee.loc[a_completer, "year"].value_counts().sort_index()
print("lignes complétées par année :", completees_par_annee.to_dict())
print("pays complétés hors 2003    :",
      sorted(pluie_completee.loc[a_completer & (pluie_completee["year"] != 2003), "area"].unique()))
print(f"rain_mm manquante           : {apres_jointures['rain_mm'].isna().sum()} -> {pluie_completee['rain_mm'].isna().sum()}")
print("pays encore sans pluie      :", sorted(pluie_completee.loc[pluie_completee["rain_mm"].isna(), "area"].unique()))

# les valeurs de pluie déjà connues ne doivent pas changer
connues = apres_jointures["rain_mm"].notna()
assert pluie_completee.loc[connues, "rain_mm"].equals(apres_jointures.loc[connues, "rain_mm"])
assert completees_par_annee.to_dict() == {1990: 3, 1991: 3, 2003: 947}

lignes complétées par année : {1990: 3, 1991: 3, 2003: 947}
pays complétés hors 2003    : ['Bahamas']
rain_mm manquante           : 1130 -> 177
pays encore sans pluie      : ['New Caledonia']


**Observations :**

- 953 lignes complétées : 947 en 2003 (année absente de `rainfall.csv`) et 6 pour les Bahamas en
  1990-1991.
- Les valeurs de pluie déjà connues ne changent pas.
- 177 lignes restent vides : la Nouvelle-Calédonie n'a aucune valeur de pluie. Elle est retirée plus
  bas, car elle n'a pas non plus de température.

## Pays sans température ou sans pesticides

Ces pays n'ont aucune valeur à reprendre : ils sont retirés.

In [14]:
par_pays = pluie_completee.groupby(["iso3", "area"])[["avg_temp", "pesticides_t", "rain_mm"]].count().reset_index()

sans_temperature = set(par_pays.loc[par_pays["avg_temp"] == 0, "iso3"])
sans_pesticides = set(par_pays.loc[par_pays["pesticides_t"] == 0, "iso3"])
dans_les_deux = sans_temperature & sans_pesticides
pays_retires = sans_temperature | sans_pesticides
sans_pluie = set(par_pays.loc[par_pays["rain_mm"] == 0, "iso3"])

print(f"pays sans température : {len(sans_temperature)}")
print(f"pays sans pesticides  : {len(sans_pesticides)}")
print(f"dans les deux groupes : {len(dans_les_deux)}")
print(f"pays retirés          : {len(pays_retires)} = {len(sans_temperature)} + {len(sans_pesticides)} - {len(dans_les_deux)}")
print(f"pays sans pluie parmi les pays sans température : {sans_pluie <= sans_temperature}")

noms = par_pays.set_index("iso3")["area"]
print("\ndans les deux groupes :", ", ".join(sorted(noms[sorted(dans_les_deux)])))
print("\npays retirés :", ", ".join(sorted(noms[sorted(pays_retires)])))

assert (len(sans_temperature), len(sans_pesticides), len(dans_les_deux), len(pays_retires)) == (36, 24, 9, 51)
assert sans_pluie <= sans_temperature

pays sans température : 36
pays sans pesticides  : 24
dans les deux groupes : 9
pays retirés          : 51 = 36 + 24 - 9
pays sans pluie parmi les pays sans température : True

dans les deux groupes : Benin, Cambodia, Cuba, Democratic People's Republic of Korea, Djibouti, Eswatini, Puerto Rico, Solomon Islands, South Sudan

pays retirés : Afghanistan, Belize, Benin, Bhutan, Bosnia and Herzegovina, Brunei Darussalam, Cambodia, Chad, Costa Rica, Cuba, Cyprus, Democratic People's Republic of Korea, Democratic Republic of the Congo, Djibouti, Equatorial Guinea, Eswatini, Ethiopia, Fiji, Gabon, Gambia, Georgia, Iceland, Israel, Jordan, Kuwait, Kyrgyzstan, Liberia, Luxembourg, Mongolia, Myanmar, New Caledonia, Nigeria, Occupied Palestinian Territory, Oman, Panama, Paraguay, Philippines, Puerto Rico, Serbia, Sierra Leone, Solomon Islands, Somalia, South Sudan, Timor-Leste, Togo, Trinidad and Tobago, Turkmenistan, United Arab Emirates, Uzbekistan, Vanuatu, Yemen


In [15]:
crop_yield_clean = pluie_completee[~pluie_completee["iso3"].isin(pays_retires)].reset_index(drop=True)

print(f"lignes : {len(pluie_completee)} -> {len(crop_yield_clean)} ({len(pluie_completee) - len(crop_yield_clean)} retirées)")
print(f"pays   : {pluie_completee['iso3'].nunique()} -> {crop_yield_clean['iso3'].nunique()}")
assert len(pluie_completee) - len(crop_yield_clean) == 6_322

lignes : 22679 -> 16357 (6322 retirées)
pays   : 168 -> 117


**Observations :**

- 36 pays sans température + 24 sans pesticides − 9 dans les deux groupes = **51 pays retirés**.
- 6 322 lignes retirées : il reste 117 pays.
- La Nouvelle-Calédonie, seul pays sans pluie, fait partie des pays sans température.

## Résultat : dataset historique nettoyé

In [16]:
print(f"lignes       : {len(crop_yield_clean)}")
print(f"clés uniques : {len(crop_yield_clean.drop_duplicates(CLE))}")
print(f"pays         : {crop_yield_clean['iso3'].nunique()}")
print(f"cultures     : {crop_yield_clean['crop'].nunique()}")
print(f"années       : {crop_yield_clean['year'].min()} - {crop_yield_clean['year'].max()} ({crop_yield_clean['year'].nunique()} distinctes)")
print(f"manquants    : {crop_yield_clean.isna().sum().sum()}")
pays_par_annee = crop_yield_clean.groupby("year")["iso3"].nunique()
print(f"pays par année : {pays_par_annee[DEBUT]} en {DEBUT}, {pays_par_annee[FIN]} en {FIN}")

assert len(crop_yield_clean) == 16_357
assert not crop_yield_clean.duplicated(CLE).any()
assert crop_yield_clean["iso3"].nunique() == 117
assert crop_yield_clean["crop"].nunique() == 10
assert (crop_yield_clean["year"].min(), crop_yield_clean["year"].max()) == (DEBUT, FIN)
assert crop_yield_clean.isna().sum().sum() == 0

lignes       : 16357
clés uniques : 16357
pays         : 117
cultures     : 10
années       : 1990 - 2013 (24 distinctes)
manquants    : 0
pays par année : 97 en 1990, 117 en 2013


In [17]:
crop_yield_clean[["year", "yield_t_ha", "avg_temp", "rain_mm", "pesticides_t"]].describe().round(2)

,year,yield_t_ha,avg_temp,rain_mm,pesticides_t
count,16357.00,16357.00,16357.00,16357.00,16357.00
mean,2001.69,6.88,19.53,1145.42,33122.37
std,6.88,7.69,6.88,713.24,154063.57
min,1990.00,0.00,1.30,51.00,0.04
25%,1996.00,1.80,14.54,619.00,207.10
50%,2002.00,3.87,20.44,1071.00,2489.75
75%,2008.00,9.20,25.93,1622.00,14485.33
max,2013.00,50.14,30.42,3240.00,1806000.00


**Observations :**

- 16 357 lignes, 117 pays, 10 cultures, 1990-2013.
- Aucune valeur manquante et une seule ligne par pays, année et culture.
- Tous les pays ne sont pas présents dès 1990 : 97 en 1990, 117 en 2013.

# Sauvegarde

Seul le dataset nettoyé est sauvegardé, dans `data/processed/crop_yield_clean.csv`. Les notebooks 05
et 06 partent de ce fichier ; l'état après jointures n'est pas sauvegardé.

Le fichier n'est pas versionné : il se reconstruit en réexécutant ce notebook.

In [18]:
PATHS.data_processed.mkdir(parents=True, exist_ok=True)
chemin_sortie = PATHS.data_processed / "crop_yield_clean.csv"

crop_yield_clean.to_csv(chemin_sortie, index=False)
# chemin relatif : pas de chemin local enregistré dans le notebook
print(f"écrit : {chemin_sortie.relative_to(PATHS.root)}")
print(f"taille : {chemin_sortie.stat().st_size / 1024**2:.1f} Mo")

écrit : data/processed/crop_yield_clean.csv
taille : 0.9 Mo


In [19]:
# Relecture du fichier écrit
relu = pd.read_csv(chemin_sortie)

print(f"lignes    : {len(relu)} (attendu {len(crop_yield_clean)})")
print(f"colonnes  : {list(relu.columns) == list(crop_yield_clean.columns)}")
print(f"doublons sur la clé : {relu.duplicated(CLE).sum()}")
print(f"manquants : {relu.isna().sum().sum()}")

assert len(relu) == len(crop_yield_clean) and list(relu.columns) == list(crop_yield_clean.columns)
assert not relu.duplicated(CLE).any() and relu.isna().sum().sum() == 0
print()
relu.head()

lignes    : 16357 (attendu 16357)
colonnes  : True
doublons sur la clé : 0
manquants : 0



,iso3,area,year,crop,yield_t_ha,avg_temp,rain_mm,pesticides_t
0,ALB,Albania,1990,Maize,3.6613,16.37,1485.0,121.0
1,ALB,Albania,1991,Maize,2.9068,15.36,1485.0,121.0
2,ALB,Albania,1992,Maize,2.4876,16.06,1485.0,121.0
3,ALB,Albania,1993,Maize,2.4185,16.05,1485.0,121.0
4,ALB,Albania,1994,Maize,2.5848,16.96,1485.0,201.0


**Observations :**

- Le fichier relu est conforme : mêmes lignes et colonnes, clé unique, aucune valeur manquante.